# Ticha HTR — manuscript-tuned and character-level language models for rescoring

`ticha_trocr_lm_fusion.ipynb` showed that the text-only word-piece decoders (`zap`, `zap_spa`) make the best end-to-end model worse, by rescoring and by shallow fusion. They were trained mostly on **printed** works (1.3M characters against 0.28M from the manuscripts), whose conventions differ from the scribes': no abbreviations such as `dho`, regular word spacing, more `ç`. And word pieces punish any spelling they haven't seen, while a quarter of the Zapotec validation words never occur in the text corpus.

This notebook tests two fixes, still **without training the HTR model**:

1. **Manuscript fine-tuning.** Each text-only decoder gets a last training stage on the manuscript **train** lines only, so it ends on the scribes' conventions. The number of epochs is chosen by validation bits per character.
2. **Character-level language models.** The character decoders from Part 4 of `ticha_trocr_finetuning-modified-transformer.ipynb`, left to right (`l2r`) and right to left (`r2l`), score the same candidates. A character model gives an unseen spelling a lower but finite score instead of a word-piece penalty. `both` averages the two directions, so each character is judged by what comes before and after it.

The candidate lines are the ones `ticha_trocr_lm_fusion.ipynb` cached: 5 per line from beam search with the end-to-end model `trocr-base-handwritten-finetuned-smooth-cosine-aug`. Each candidate's score is `(log p_HTR + β · log p_LM) / tokens`, as in that notebook; `β = 0` is the HTR model alone (validation CER 0.161, test CER 0.144 without Za687). `β` is chosen on validation and the test split is scored once at the end.

**Run it on the VM** (`ticha-l4`) with the **Python (ticha-trocr)** kernel: the fine-tuning needs about 6 GB of GPU memory, and the base decoders live in `~/ticha-runs/trocr-modified/`. Fine-tuned decoders go to `~/ticha-runs/lm-manuscript/`, and are loaded instead of trained again when they exist. The candidate files must be in `data/evaluation/lm_fusion/` (copy them from the Mac if they are missing; the first cell prints the command).

In [ ]:
import os
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import copy
import json
import math
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from jiwer import cer, process_characters, wer
from transformers import TrOCRProcessor, VisionEncoderDecoderConfig, get_linear_schedule_with_warmup
from transformers.models.trocr.modeling_trocr import TrOCRForCausalLM

# The kernel may start in the repo root (the VM's Jupyter service) or in notebooks/train/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').exists())
RUNS = Path(os.environ.get('TICHA_RUNS_DIR', '~/ticha-runs')).expanduser()
BASE_DIR = RUNS / 'trocr-modified'      # text-only decoders from the modified-transformer notebook
TUNED_DIR = RUNS / 'lm-manuscript'      # this notebook's manuscript-tuned decoders
NBEST_DIR = ROOT / 'data' / 'evaluation' / 'lm_fusion'
OUTPUT_DIR = ROOT / 'data' / 'evaluation' / 'lm_manuscript'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = 'microsoft/trocr-base-handwritten'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
AMP = dict(device_type='cuda', dtype=torch.bfloat16) if DEVICE.type == 'cuda' else dict(device_type='cpu', enabled=False)
VARIANTS = ['zap', 'zap_spa']
DIRECTIONS = ['l2r', 'r2l']
LM_FLOOR = -12.0          # lowest log-probability per token, as in ticha_trocr_lm_fusion.ipynb
FT_EPOCHS = 6             # most epochs of manuscript fine-tuning; the best one on validation is kept
FT_LR = 2e-5              # below the 5e-5 of the first text stage, so the printed-corpus knowledge isn't erased
FT_BATCH_SIZE = 32
SCORE_BATCH_SIZE = 64
BETAS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
COMPARABLE_EXCLUDE = {'Za687'}  # same pages as Th687, which the end-to-end model was trained on
SEED = 42

NBEST_FILES = {split: NBEST_DIR / f'nbest_{split}_start2.json' for split in ['validation', 'test']}
missing = [str(p) for p in NBEST_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Candidate files missing: ' + ', '.join(missing) + '\nRun ticha_trocr_lm_fusion.ipynb first, or copy them from the Mac:\n'
        '  gcloud compute scp --project dml-course-511723 --zone us-central1-a '
        'data/evaluation/lm_fusion/nbest_*_start2.json ticha-l4:/home/paw/ticha/data/evaluation/lm_fusion/')
print('Device:', DEVICE, '| repo:', ROOT, '| base decoders:', BASE_DIR)

## Manuscript lines

The same cleanup as in the training notebooks: abbreviation expansions and glosses removed, damaged or illegible lines dropped, uncertain readings kept as text in train and dropped from validation and test. Th687 is dropped from train, as there: it is the same manuscript as Za687 (test).

Fine-tuning text, as in the first text stage:

| Decoder | Manuscript lines |
|---|---|
| `zap` | train lines from Zapotec documents |
| `zap_spa` | all train lines |

Validation lines measure each decoder (bits per character, per language) and choose the number of epochs. They are also the lines `β` is chosen on, so a decoder is never trained on them.

In [ ]:
BRACKET_RE = re.compile(r'\[([^\[\]]*)\]|\(([^()]*)\)')
NOTE_RE = re.compile(
    r'[?…]|^\.+$|\.\.|\s|,|^x+$|^word$'
    r'|cortado|roto|torn|mancha|borrado|firmas|ilegible',
    re.IGNORECASE)
DROP_HASH = True
GUESS_RE = re.compile(r'\((?!word\?)([^()?]*[^\W\d_][^()?]*)\?+\)')


def clean_text(text, keep_guesses=False):
    """Return the diplomatic text, or None when the line should be dropped."""
    text = str(text or '')
    if keep_guesses:
        text = GUESS_RE.sub(lambda m: m.group(1), text)
    notes = [square or round for square, round in BRACKET_RE.findall(text)]
    if any(NOTE_RE.search(note.strip()) for note in notes if not note.strip().startswith('=')):
        return None
    text = BRACKET_RE.sub('', text)
    if re.search(r'[\[\]()]', text) or (DROP_HASH and '#' in text):
        return None
    return re.sub(r'\s+', ' ', text).strip() or None


ds = load_dataset('maclenn77/ticha-htr-lines', 'original')
lines = {}
for split in ['train', 'validation', 'test']:
    frame = ds[split].remove_columns('image').to_pandas()
    frame['text'] = frame['text'].map(lambda t: clean_text(t, keep_guesses=split == 'train'))
    lines[split] = frame[frame['text'].notna()].reset_index(drop=True)
    print(f'{split}: kept {len(lines[split]):,} / {len(frame):,} lines')


def line_key(text):
    return re.sub(r'\W+', ' ', text.lower()).strip()


# A train document is dropped when more than half of its lines appear in validation or test (Th687).
eval_keys = {line_key(t) for split in ['validation', 'test'] for t in lines[split]['text'] if len(t) >= 15}
share = lines['train'].groupby('doc_id')['text'].agg(lambda texts: np.mean([line_key(t) in eval_keys for t in texts]))
duplicate_docs = set(share[share > 0.5].index)
lines['train'] = lines['train'][~lines['train']['doc_id'].isin(duplicate_docs)].reset_index(drop=True)
print('Train documents dropped as duplicates of evaluation documents:', sorted(duplicate_docs),
      f"-> train: {len(lines['train']):,} lines")

finetune_texts = {
    'zap': list(lines['train'].loc[lines['train']['language'] == 'Zapotec', 'text']),
    'zap_spa': list(lines['train']['text']),
}
val = lines['validation']
lm_eval_texts = {'zap': list(val.loc[val['language'] == 'Zapotec', 'text']),
                 'spa': list(val.loc[val['language'] != 'Zapotec', 'text'])}
for name, texts in finetune_texts.items():
    print(f'Fine-tuning text {name}: {len(texts):,} lines, {sum(map(len, texts)):,} characters')
print({name: len(texts) for name, texts in lm_eval_texts.items()}, 'validation lines per language')

nbest = {split: json.loads(path.read_text()) for split, path in NBEST_FILES.items()}
for split, rows in nbest.items():
    known = set(lines[split]['id'])
    assert all(row['id'] in known for row in rows), f'{split}: candidates for lines the cleanup dropped'
    print(f'{split}: {len(rows)} lines with {len(rows[0]["candidates"])} candidates each')

## The language models

All of them are the TrOCR decoder used as a plain language model (`TrOCRForCausalLM` without cross-attention inputs), as in training:

- **Word pieces** (`wp zap`, `wp zap_spa`): RoBERTa's tokenizer, as the HTR model, with the vocabulary mask from training. A candidate is scored on the HTR model's own token ids, exactly as in `ticha_trocr_lm_fusion.ipynb`.
- **Characters** (`char zap l2r`, …): one id per character; the vocabulary is RoBERTa's four special tokens plus the characters of the cleaned train lines (the printed text was reduced to those characters), which gives the 112 ids of the saved models. `r2l` reads the line reversed. A character the vocabulary lacks becomes `<unk>`.

Every log-probability is floored at `LM_FLOOR` per token, so a single unseen piece can't veto a line. **Bits per character** (validation, lower is better) put both kinds on the same scale: the line's total log-probability, `</s>` included, divided by its characters plus one.

In [ ]:
tokenizer = TrOCRProcessor.from_pretrained(MODEL_ID).tokenizer
DECODER_CONFIG = VisionEncoderDecoderConfig.from_pretrained(MODEL_ID).decoder
BOS, PAD, EOS, UNK = 0, 1, 2, 3  # RoBERTa's ids; the character vocabulary keeps them
CHAR_VOCAB = ['<s>', '<pad>', '</s>', '<unk>'] + sorted(set(''.join(lines['train']['text'])))
CHAR_ID = {ch: i for i, ch in enumerate(CHAR_VOCAB)}
MAX_TOKENS = 256


def lm_specs():
    """(name, kind, variant, direction) for every base language model."""
    specs = [(f'wp {v}', 'wp', v, None) for v in VARIANTS]
    specs += [(f'char {v} {d}', 'char', v, d) for v in VARIANTS for d in DIRECTIONS]
    return specs


def base_path(kind, variant, direction):
    folder = f'decoder_{variant}' if kind == 'wp' else f'char_decoder_{variant}_{direction}'
    return BASE_DIR / folder / 'decoder_state.pt'


def tuned_path(name):
    return TUNED_DIR / name.replace(' ', '_') / 'decoder_state.pt'


def load_lm(path):
    state = torch.load(path, map_location='cpu', weights_only=True)
    mask = state.pop('output_projection.vocab_mask', None)
    config = copy.deepcopy(DECODER_CONFIG)
    config.vocab_size = state['model.decoder.embed_tokens.weight'].shape[0]
    lm = TrOCRForCausalLM(config)
    lm.load_state_dict(state)
    lm.vocab_mask = (mask if mask is not None else torch.zeros(config.vocab_size)).to(DEVICE)
    return lm.to(DEVICE).eval()


def save_lm(lm, path):
    state = {k: v.detach().cpu() for k, v in lm.state_dict().items()}
    if lm.vocab_mask.any():
        state['output_projection.vocab_mask'] = lm.vocab_mask.cpu()
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def text_labels(kind, direction, text):
    """Target ids for a text: what the language model predicts after its start token."""
    if kind == 'wp':
        return tokenizer(text).input_ids[:MAX_TOKENS]  # <s> ... </s>, as in the first text stage
    text = text[::-1] if direction == 'r2l' else text
    return [CHAR_ID.get(ch, UNK) for ch in text][:MAX_TOKENS - 1] + [EOS]


def candidate_labels(kind, direction, candidate):
    """Word pieces: the HTR model's own ids after its start token, up to the first </s>. Characters: the text."""
    if kind == 'char':
        return text_labels(kind, direction, candidate['text'])
    ids = candidate['ids'][1:]
    return ids[:ids.index(EOS) + 1] if EOS in ids else ids


def batch(label_lists):
    width = max(map(len, label_lists))
    labels = torch.full((len(label_lists), width), -100, dtype=torch.long)
    for row, ids in enumerate(label_lists):
        labels[row, :len(ids)] = torch.tensor(ids)
    inputs = torch.cat([torch.full((len(label_lists), 1), BOS), labels[:, :-1]], dim=1)
    return inputs.masked_fill(inputs == -100, PAD).to(DEVICE), labels.to(DEVICE)


def log_probs(lm, inputs):
    with torch.autocast(**AMP):
        logits = lm(input_ids=inputs).logits
    return (logits.float() + lm.vocab_mask).log_softmax(-1)


@torch.inference_mode()
def sequence_log_probs(lm, label_lists, floor=LM_FLOOR):
    """Summed (floored) log-probability of each label sequence."""
    totals = []
    for start in range(0, len(label_lists), SCORE_BATCH_SIZE):
        inputs, labels = batch(label_lists[start:start + SCORE_BATCH_SIZE])
        token_logp = log_probs(lm, inputs).clamp(min=floor).gather(-1, labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
        totals += (token_logp * (labels != -100)).sum(-1).tolist()
    return totals


def bits_per_char(lm, kind, direction, texts):
    total = sum(sequence_log_probs(lm, [text_labels(kind, direction, t) for t in texts]))
    return -total / sum(len(t) + 1 for t in texts) / math.log(2)


print(f'Character vocabulary: {len(CHAR_VOCAB)} ids')

## 1. Manuscript fine-tuning

Each base decoder is trained further on its manuscript lines (`FT_EPOCHS` at most, learning rate `FT_LR`, linear decay, cross-attention frozen as in the first text stage). After each epoch the validation bits per character are measured on the lines of the decoder's language (Zapotec lines for `zap`, all lines for `zap_spa`), and the best epoch is kept; epoch 0 is the base decoder, so fine-tuning can't make it worse on validation.

The table compares bits per character before and after, by language. A decoder that fits the scribes better should drop most on the Zapotec lines.

In [ ]:
def finetune(name, kind, variant, direction):
    path, history_path = tuned_path(name), tuned_path(name).with_name('history.json')
    if path.exists() and history_path.exists():
        print(f'{name}: using the saved decoder {path}')
        return json.loads(history_path.read_text())
    torch.manual_seed(SEED)
    lm = load_lm(base_path(kind, variant, direction))
    for param_name, param in lm.named_parameters():
        param.requires_grad = 'encoder_attn' not in param_name
    texts = finetune_texts[variant]
    select_texts = lm_eval_texts['zap'] if variant == 'zap' else lm_eval_texts['zap'] + lm_eval_texts['spa']
    params = [p for p in lm.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=FT_LR, weight_decay=0.01)
    steps = FT_EPOCHS * math.ceil(len(texts) / FT_BATCH_SIZE)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * steps), steps)

    def evaluate(epoch, loss=None):
        lm.eval()
        row = {'epoch': epoch, 'train_loss': loss, 'select_bpc': bits_per_char(lm, kind, direction, select_texts)}
        row.update({f'{lang}_bpc': bits_per_char(lm, kind, direction, t) for lang, t in lm_eval_texts.items()})
        print(f"{name} epoch {epoch}: " + ', '.join(f'{k} {v:.3f}' for k, v in row.items() if k != 'epoch' and v is not None), flush=True)
        return row

    history = [evaluate(0)]
    best_bpc, best_state = history[0]['select_bpc'], None
    rng = np.random.default_rng(SEED)
    for epoch in range(1, FT_EPOCHS + 1):
        lm.train()
        order, losses = rng.permutation(len(texts)), []
        for start in range(0, len(texts), FT_BATCH_SIZE):
            inputs, labels = batch([text_labels(kind, direction, texts[i]) for i in order[start:start + FT_BATCH_SIZE]])
            with torch.autocast(**AMP):
                logits = lm(input_ids=inputs).logits
            loss = torch.nn.functional.cross_entropy((logits.float() + lm.vocab_mask).transpose(1, 2), labels, ignore_index=-100)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())
        history.append(evaluate(epoch, float(np.mean(losses))))
        if history[-1]['select_bpc'] < best_bpc:
            best_bpc = history[-1]['select_bpc']
            best_state = {k: v.detach().cpu().clone() for k, v in lm.state_dict().items()}
    if best_state is not None:
        lm.load_state_dict(best_state)
    best_epoch = min(history, key=lambda row: row['select_bpc'])['epoch']
    save_lm(lm, path)
    result = {'best_epoch': best_epoch, 'history': history}
    history_path.write_text(json.dumps(result))
    print(f'{name}: kept epoch {best_epoch}, saved to {path}')
    del lm, optimizer
    torch.cuda.empty_cache()
    return result


t0 = time.time()
finetuned = {name: finetune(name, kind, variant, direction) for name, kind, variant, direction in lm_specs()}
print(f'Fine-tuning done in {(time.time() - t0) / 60:.1f} min')

bpc = pd.DataFrame([{'lm': name, 'kept epoch': result['best_epoch'],
                     'Zapotec before': result['history'][0]['zap_bpc'],
                     'Zapotec after': result['history'][result['best_epoch']]['zap_bpc'],
                     'Spanish before': result['history'][0]['spa_bpc'],
                     'Spanish after': result['history'][result['best_epoch']]['spa_bpc']}
                    for name, result in finetuned.items()]).set_index('lm')
print('Validation bits per character (lower is better), before and after manuscript fine-tuning')
display(bpc.round(3))

## 2. Score the candidates

Every candidate line gets the summed log-probability of each language model: the six base decoders and their six manuscript-tuned versions (` +ms`). The scores are cached in `data/evaluation/lm_manuscript/`, so a rerun doesn't score again. As a check, the base word-piece scores must match the ones `ticha_trocr_lm_fusion.ipynb` stored.

In [ ]:
SCORES_FILES = {split: OUTPUT_DIR / f'scores_{split}.json' for split in nbest}
scored = {split: json.loads(path.read_text()) if path.exists() else copy.deepcopy(nbest[split])
          for split, path in SCORES_FILES.items()}
lm_paths = {}
for name, kind, variant, direction in lm_specs():
    lm_paths[name] = (kind, direction, base_path(kind, variant, direction))
    lm_paths[name + ' +ms'] = (kind, direction, tuned_path(name))

for lm_name, (kind, direction, path) in lm_paths.items():
    key = 'lm ' + lm_name
    if all(key in c for rows in scored.values() for row in rows for c in row['candidates']):
        continue
    lm = load_lm(path)
    for split, rows in scored.items():
        candidates = [c for row in rows for c in row['candidates']]
        for c, value in zip(candidates, sequence_log_probs(lm, [candidate_labels(kind, direction, c) for c in candidates])):
            c[key] = value
        SCORES_FILES[split].write_text(json.dumps(rows))
    print('Scored with', lm_name, flush=True)
    del lm
    torch.cuda.empty_cache()

for variant in VARIANTS:
    gap = max(abs(c[variant] - c[f'lm wp {variant}']) for rows in scored.values() for row in rows for c in row['candidates'])
    print(f'wp {variant}: largest difference from the stored lm_fusion score {gap:.3f} nats (bf16 rounding is expected)')

## Choose β on validation

Each configuration is a language model (or `both` directions of a character model, averaged) with a weight `β`. The table gives validation CER for every `β`; `β = 0` is the HTR model alone. The configuration with the lowest CER is the one taken to the test split; the others are shown on test too, each with its own validation `β`, but only the first one counts as chosen in advance.

In [ ]:
def lm_score(candidate, config):
    """Log-probability of a candidate under a configuration; `both` averages the two character directions."""
    if config.endswith(' both') or config.endswith(' both +ms'):
        suffix = ' +ms' if config.endswith('+ms') else ''
        base = config.replace(' both', '').replace(' +ms', '')
        return 0.5 * (candidate[f'lm {base} l2r{suffix}'] + candidate[f'lm {base} r2l{suffix}'])
    return candidate['lm ' + config]


CONFIGS = [f'{kind} {v}{d}{ms}' for ms in ['', ' +ms'] for v in VARIANTS
           for kind, d in [('wp', ''), ('char', ' l2r'), ('char', ' r2l'), ('char', ' both')]]


def pick(rows, config=None, beta=0.0):
    if not config or not beta:
        return [row['candidates'][0]['text'] for row in rows]
    return [max(row['candidates'], key=lambda c: (c['htr'] + beta * lm_score(c, config)) / max(c['tokens'], 1))['text']
            for row in rows]


references = {split: dict(zip(lines[split]['id'], lines[split]['text'])) for split in scored}
languages = {split: dict(zip(lines[split]['id'], lines[split]['language'])) for split in scored}


def split_cer(split, predictions, keep=lambda line_id: True):
    rows = [(references[split][row['id']], p) for row, p in zip(scored[split], predictions) if keep(row['id'])]
    return cer([r for r, _ in rows], [p for _, p in rows])


sweep = pd.DataFrame({beta: {config: split_cer('validation', pick(scored['validation'], config, beta)) for config in CONFIGS}
                      for beta in BETAS})
sweep.columns.name, sweep.index.name = 'beta', 'configuration'
print('Validation CER (rows: language model; columns: beta). beta = 0 is the HTR model alone.')
display(sweep.round(4))

best_beta = sweep.idxmin(axis=1)
zapotec_only = {config: split_cer('validation', pick(scored['validation'], config, best_beta[config]),
                                  lambda i: languages['validation'][i] == 'Zapotec') for config in CONFIGS}
summary = pd.DataFrame({'best beta': best_beta, 'validation CER': sweep.min(axis=1),
                        'validation CER, Zapotec lines': pd.Series(zapotec_only)})
baseline_val = sweep[0.0].iloc[0]
baseline_zap = split_cer('validation', pick(scored['validation']), lambda i: languages['validation'][i] == 'Zapotec')
oracle_val = cer([references['validation'][row['id']] for row in scored['validation']],
                 [min(row['candidates'], key=lambda c: cer(references['validation'][row['id']], c['text'] or ' '))['text']
                  for row in scored['validation']])
print(f'HTR alone: {baseline_val:.4f} (Zapotec lines {baseline_zap:.4f}) | best of the 5 candidates (oracle): {oracle_val:.4f}')
display(summary.sort_values('validation CER').round(4))
CHOSEN = summary['validation CER'].idxmin()
print(f'Chosen on validation: {CHOSEN}, beta={best_beta[CHOSEN]}' if best_beta[CHOSEN] else
      'No language model beats the HTR model alone on validation.')

## Test evaluation

Each configuration with its validation `β`, against the HTR model alone. Za687 is left out (the end-to-end model was trained on its duplicate Th687), so these numbers are comparable with the 0.144 of `ticha_trocr_lm_fusion.ipynb`. The difference between the chosen configuration and the HTR model alone gets a 95% interval from resampling whole documents.

In [ ]:
test_keep = lambda line_id: line_id.split('_')[0] not in COMPARABLE_EXCLUDE
test_rows = [row for row in scored['test'] if test_keep(row['id'])]
test_ids = [row['id'] for row in test_rows]
test_refs = [references['test'][i] for i in test_ids]
test_langs = pd.Series([languages['test'][i] for i in test_ids])

predictions = {'HTR alone': pick(test_rows)}
for config in CONFIGS:
    if best_beta[config]:
        predictions[f'{config} (beta={best_beta[config]})'] = pick(test_rows, config, best_beta[config])

table = []
for name, preds in predictions.items():
    row = {'model': name, 'all': cer(test_refs, preds)}
    for lang in sorted(test_langs.unique()):
        idx = np.where(test_langs == lang)[0]
        row[lang] = cer([test_refs[i] for i in idx], [preds[i] for i in idx])
    row['WER'] = wer(test_refs, preds)
    table.append(row)
test_table = pd.DataFrame(table).set_index('model')
print(f'Test CER without Za687 ({len(test_rows)} lines)')
display(test_table.round(4))
test_table.to_csv(OUTPUT_DIR / 'test_scores_without_Za687.csv')
pd.DataFrame({'id': test_ids, 'language': test_langs, 'reference': test_refs, **predictions}).to_csv(
    OUTPUT_DIR / 'test_predictions_without_Za687.csv', index=False)


def char_errors(refs, preds):
    out = []
    for r, p in zip(refs, preds):
        o = process_characters(r, p or ' ')
        out.append(o.substitutions + o.deletions + o.insertions)
    return np.array(out)


if best_beta[CHOSEN]:
    chosen_name = f'{CHOSEN} (beta={best_beta[CHOSEN]})'
    lengths = np.array([len(r) for r in test_refs])
    base_err, new_err = char_errors(test_refs, predictions['HTR alone']), char_errors(test_refs, predictions[chosen_name])
    docs = np.array([i.split('_')[0] for i in test_ids])
    by_doc = {d: np.where(docs == d)[0] for d in np.unique(docs)}
    rng = np.random.default_rng(SEED)
    diffs = []
    for _ in range(5000):
        s = np.concatenate([by_doc[d] for d in rng.choice(list(by_doc), len(by_doc))])
        diffs.append((new_err[s].sum() - base_err[s].sum()) / lengths[s].sum())
    print(f'{chosen_name} minus HTR alone: {(new_err.sum() - base_err.sum()) / lengths.sum():+.4f} CER, '
          f'95% interval by document [{np.percentile(diffs, 2.5):+.4f}, {np.percentile(diffs, 97.5):+.4f}], '
          f'better in {np.mean(np.array(diffs) < 0):.1%} of resamples')

In [ ]:
# Validation lines the chosen configuration changed, to see what it fixes and what it breaks.
if best_beta[CHOSEN]:
    changed = pd.DataFrame({'reference': [references['validation'][row['id']] for row in scored['validation']],
                            'HTR alone': pick(scored['validation']),
                            CHOSEN: pick(scored['validation'], CHOSEN, best_beta[CHOSEN])})
    changed = changed[changed['HTR alone'] != changed[CHOSEN]]
    changed['CER change'] = [cer(r, b or ' ') - cer(r, a or ' ') for r, a, b in
                             zip(changed['reference'], changed['HTR alone'], changed[CHOSEN])]
    print(f'{CHOSEN}: {len(changed)} of {len(scored["validation"])} validation lines changed, '
          f'{(changed["CER change"] < 0).sum()} better, {(changed["CER change"] > 0).sum()} worse')
    pd.set_option('display.max_colwidth', 80)
    display(changed.sort_values('CER change').head(12))
    display(changed.sort_values('CER change').tail(12))

## How to read the results

- **Bits per character** say whether manuscript fine-tuning made the decoders better models of the scribes' text, before any HTR. If they don't drop on the Zapotec validation lines, the rescoring can't improve either.
- **Word pieces against characters:** compare `wp …` with `char … both` at the same stage. Characters should suffer less from unseen Zapotec spellings.
- **One direction against two:** `char … l2r` and `char … r2l` against `char … both`.
- **The ceiling** is the oracle line: the CER if the best of the 5 candidates were always picked. Rescoring can't go below it.
- Only the configuration chosen on validation is a fair test result; the other test rows are for discussion.